# MRIxFields Etapa 2 — Gate 0 (diagnóstico)

**La pregunta ya no es "¿reconstruye Stage 1?"** — run C la contestó. Es:

> ¿Stage 2 está aprendiendo traducción condicional de campo desde distribuciones no pareadas,
> o mayormente un atajo de intensidad/estadística?

Lo que midió v2 en el traveller held-out 0006 (60 pares, decode full-volume):

| | nRMSE | SSIM |
|---|---|---|
| transport (SB v2) | 0.459 | 0.880 |
| identity | 0.595 | 0.876 |
| ceiling (VAE congelado) | 0.131 | 0.966 |

nRMSE se movió mucho, SSIM nada contra un techo de 0.966. Firma de reescalado global de
intensidad, no de estructura.

## Cómo leer los baselines (asimetría importante)

El afín **latente** prueba un mecanismo estrecho. Su forma cerrada es exacta solo bajo una
aproximación gaussiana 1-D por canal, y los canales reales son espaciales, no gaussianos y se
decodifican de forma no lineal. Entonces:

- afín latente **≈** SB → evidencia fuerte de atajo.
- afín latente **≠** SB → **no prueba** que SB aprendió física de campo. Descarta un afín por
  canal en latente y nada más.

Por eso el gate incluye también baselines en **espacio imagen** (afín robusto e histogram
matching) aplicados a la reconstrucción identity y ajustados **solo con sujetos de training**.
El histogram matching es el mapa puramente fotométrico más fuerte que existe: reproduce toda la
distribución de intensidad del dominio destino sin tocar estructura. Lo que un modelo gane sobre
él, por construcción, no es fotométrico.

## Orden de ejecución

Barato primero, caro solo si queda una pregunta sin resolver. El paso caro está detrás de
`RUN_EXPENSIVE_REFERENCE = False`.

1. Provenance + validación de banco full + hash del VAE.
2. Baselines afín (latente) e intensidad (imagen).
3. Wrong-target sweep — responde **y** en qué dirección.
4. Clasificación de dominio destino (dentro del sweep).
5. Calidad del coupling por contraste.
6. Residual — explícitamente descriptivo.
7. Solo entonces: decode completo + LPIPS.

**Protocolo de sujetos:** desarrolla y depura en **0007** (training traveller). Congela código,
umbrales y formato de reporte. Evalúa **una sola vez** en 0006. **0009 no se toca.**

## 1. Entorno, rutas y contrato de provenance

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

DRIVE = "/content/drive/MyDrive/MRIxFields2026"

# --- ENTRADAS verificadas --------------------------------------------------------------------
LATENTS  = f"{DRIVE}/LatentBanks/runC_ep015_step047700_74132b9c_full_bf16"
SPLIT    = f"{DRIVE}/split_v3.json"
VAE_CKPT = f"{DRIVE}/vae_kl_vae_best.pt"
SB_CKPT  = (
    f"{DRIVE}/Runs/stage2_gate_runC_74132b9c_c4b9c39_full/"
    "sb_v2/ckpt/transport_sb_brownian_v2_last.pt"
)
EXPECTED_VAE_SHA256 = "74132b9c514bb91b86d8eb43c63542780bce11304e31e67d3bf75c90ff5d4d79"

# El commit exacto del diagnóstico. Se hace checkout --detach a este sha.
DIAGNOSTIC_SHA = "d3476b900866019b428d52d01a6d5b26b93ca65d"

# Gate del paso caro. Corre lo barato, inspecciona, y enciéndelo a propósito.
RUN_EXPENSIVE_REFERENCE = False

# Sujeto a evaluar. Desarrolla en 0007; cambia a 0006 UNA vez, con todo congelado.
EVAL_SUBJECT = "0007"
# ---------------------------------------------------------------------------------------------

VAE_CONFIG = "configs/experiment/stage1_vae_v2_fgw_freebits.yaml"
SB_CONFIG  = "configs/experiment/stage2_transport_sb_v2.yaml"

# WORK versionado por el commit del diagnóstico: dos versiones del código nunca escriben
# resultados en el mismo sitio.
WORK = f"{DRIVE}/stage2_gate0/{DIAGNOSTIC_SHA[:12]}"
Path(WORK).mkdir(parents=True, exist_ok=True)

for label, path in [("LATENTS", LATENTS), ("SPLIT", SPLIT), ("VAE_CKPT", VAE_CKPT)]:
    assert Path(path).exists(), f"STOP: {label} no existe -> {path}"
    print(f"{label:10s} OK  {path}")

HAS_SB = Path(SB_CKPT).is_file()
print(f"{'SB_CKPT':10s} {'OK' if HAS_SB else 'AUSENTE'}  {SB_CKPT}")
print(f"\nWORK: {WORK}\nEVAL_SUBJECT: {EVAL_SUBJECT}   RUN_EXPENSIVE_REFERENCE: {RUN_EXPENSIVE_REFERENCE}")

In [ ]:
import hashlib

def sha256_of(path, chunk=1 << 22):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()

vae_sha = sha256_of(VAE_CKPT)
print("VAE sha256:", vae_sha)
assert vae_sha == EXPECTED_VAE_SHA256, (
    f"STOP: el VAE no es el esperado.\n  got      {vae_sha}\n  expected {EXPECTED_VAE_SHA256}\n"
    "El banco de latentes fue construido con ese checkpoint; otro decoder invalida todo."
)
print("VAE checkpoint verificado contra el hash del banco.")

## 2. Repositorio pinneado al commit del diagnóstico

`checkout --detach` al sha exacto. Sin esto, un avance de la rama cambia el código bajo los
resultados sin dejar rastro.

In [ ]:
import shutil
import subprocess
from pathlib import Path

REPO = Path("/content/MRIxFields")
assert DIAGNOSTIC_SHA != "REPLACE_WITH_COMMIT_SHA", "STOP: fija DIAGNOSTIC_SHA."

if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "https://github.com/GuillermoTafoya/MRIxFields.git", str(REPO)],
               check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", DIAGNOSTIC_SHA], check=True)
HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert HEAD == DIAGNOSTIC_SHA, f"STOP: HEAD={HEAD} != {DIAGNOSTIC_SHA}"
print("HEAD:", HEAD)

%cd /content/MRIxFields
!pip -q install -e ".[nifti,evaluation,official-evaluation]" scipy

for command in ("fit-affine-baseline", "fit-intensity-baseline", "gate0-sweep",
                "gate0-coupling-quality", "gate0-reference-gate", "gate0-residual-gate",
                "gate0-report"):
    subprocess.run(["python", "-m", "fieldbridge.cli", command, "--help"],
                   check=True, stdout=subprocess.DEVNULL)
print("CLI Gate-0 verificado.")

## 3. Resplit, guard del banco y contrato de provenance

El resplit recomputa los fingerprints de membresía. Antes los heredaba del archivo de entrada, y
como son sensibles a membresía el resultado era ilegible para `load_vae_splits` ("stale or
altered") — bug que bloqueaba silenciosamente que el gate usara su anchor held-out.

El contrato guarda todos los hashes en un JSON. Cualquier resultado posterior se puede atar a
exactamente qué código, checkpoint, split, banco y config lo produjeron.

In [ ]:
import json
from pathlib import Path

from fieldbridge.data.resplit import resplit_file
from fieldbridge.data.vae_splits import load_vae_splits
from fieldbridge.evaluation.stage2_gate0 import assert_full_volume_bank

SPLIT_RESPLIT = f"{WORK}/split_v4_111.json"
if not Path(SPLIT_RESPLIT).exists():
    summary = resplit_file(SPLIT, SPLIT_RESPLIT, ["P:0006"], "validation")
    assert summary["counts"] == {"train": 1575, "validation": 204, "test": 205}, \
        f"STOP: conteos inesperados -> {summary['counts']}"

splits = load_vae_splits(SPLIT_RESPLIT)
travellers = lambda rs: sorted({r.subject_id for r in rs if str(r.case_id).startswith("P_")})
print("travellers  train:", travellers(splits.train),
      " validation:", travellers(splits.validation),
      " test:", travellers(splits.test))
assert travellers(splits.train) == ["0007"] and travellers(splits.validation) == ["0006"]
assert EVAL_SUBJECT in ("0006", "0007"), "0009 está congelado."

manifest = json.loads((Path(LATENTS) / "latent_bank_manifest.json").read_text())
bank_provenance = assert_full_volume_bank(manifest)

CONTRACT = {
    "diagnostic_commit": HEAD,
    "eval_subject": EVAL_SUBJECT,
    "run_expensive_reference": RUN_EXPENSIVE_REFERENCE,
    "vae_checkpoint": {"path": VAE_CKPT, "sha256": vae_sha},
    "sb_checkpoint": {"path": SB_CKPT, "sha256": sha256_of(SB_CKPT) if HAS_SB else None},
    "split": {"source": SPLIT, "resplit": SPLIT_RESPLIT,
              "fingerprint": json.loads(Path(SPLIT_RESPLIT).read_text())["fingerprint"]},
    "latent_bank": {"path": LATENTS, **bank_provenance},
    "configs": {name: sha256_of(name) for name in (VAE_CONFIG, SB_CONFIG,
                                                   "configs/experiment/stage2_gate0.yaml")},
}
Path(f"{WORK}/gate0_contract.json").write_text(json.dumps(CONTRACT, indent=2, sort_keys=True))
print(json.dumps(CONTRACT, indent=2, sort_keys=True))

## 4. Baselines: afín latente + intensidad en imagen

El afín latente es cerrado (matching de momentos por canal, que es el transporte óptimo 1-D
entre las gaussianas con esos momentos). Los de intensidad se ajustan **solo con sujetos de
training** — una referencia construida con el traveller evaluado le regalaría la respuesta.

~10-15 min el afín (una pasada de ~11 GB), pocos minutos el de intensidad.

In [ ]:
AFFINE_STEM = f"{WORK}/affine_baseline.json"
AFFINE_ALL  = f"{WORK}/affine_baseline_all.json"
AFFINE_FG   = f"{WORK}/affine_baseline_foreground.json"

if not Path(AFFINE_ALL).is_file():
    subprocess.run(["python", "-u", "-m", "fieldbridge.cli", "fit-affine-baseline",
                    "--bank-dir", LATENTS, "--split-json", SPLIT_RESPLIT,
                    "--pool-split", "train", "--cohort", "R",
                    "--out", AFFINE_STEM, "--log-every", "200"], check=True)

INTENSITY_STEM = f"{WORK}/intensity_baseline.json"
INTENSITY_HIST = f"{WORK}/intensity_baseline_histogram.json"
INTENSITY_AFF  = f"{WORK}/intensity_baseline_robust_affine.json"

if not Path(INTENSITY_HIST).is_file():
    subprocess.run(["python", "-u", "-m", "fieldbridge.cli", "fit-intensity-baseline",
                    "--split-json", SPLIT_RESPLIT, "--out", INTENSITY_STEM,
                    "--per-domain", "8", "--device", "cuda"], check=True)

from fieldbridge.models.translators.affine_baseline import AffineLatentBaseline
from fieldbridge.data.domains import Contrast, Domain

baseline = AffineLatentBaseline.load(AFFINE_FG)
print(f"\n{'par':22s} {'a (por canal)':34s} {'b (por canal)'}")
for f_s, f_t in [(0.1, 7.0), (0.1, 1.5), (3.0, 7.0), (7.0, 0.1)]:
    a, b = baseline.coefficients(Domain(f_s, Contrast.T1W), Domain(f_t, Contrast.T1W))
    print(f"T1w {f_s:>4g}T->{f_t:<4g}T   {[round(float(v), 3) for v in a]}   {[round(float(v), 3) for v in b]}")
print("\nMedido en CPU sobre este banco: a ~ 1, b ~ 0. El afín cerrado es casi la identidad EN")
print("LATENTE, asi que el reescalado de intensidad de v2 NO es un afin por canal en latente.")
print("Por eso los baselines de imagen de abajo son los que de verdad prueban la hipotesis.")

## 5. Wrong-target sweep: ¿responde, y **hacia dónde**?

Dos preguntas que no se mezclan:

- **Responsiveness** — ¿la salida cambia con el campo pedido? Spread de las 5 salidas contra el
  spread de los 5 destinos reales.
- **Dirección** — ¿cambia *correctamente*? Cada salida se asigna al más cercano de los 5
  latentes destino reales **de ese mismo sujeto y contraste**: un clasificador de dominio
  destino sin entrenamiento y sin datos extra. Misma anatomía en todos los candidatos, así que
  lo único que los distingue es el campo. Chance = 1/5 = 0.20.

Responsiveness sin dirección es un modelo que reacciona a su conditioning moviéndose a cualquier
lado, que no es traducción. Todo en latente, sin decode.

In [ ]:
SWEEP_OUT = f"{WORK}/gate0_sweep_{EVAL_SUBJECT}.json"

if HAS_SB:
    command = ["python", "-u", "-m", "fieldbridge.cli", "gate0-sweep",
               "--bank-dir", LATENTS, "--split-json", SPLIT_RESPLIT,
               "--subjects", EVAL_SUBJECT,
               "--transport-config", SB_CONFIG, "--transport-checkpoint", SB_CKPT,
               "--solver", "heun", "--n-steps", "20",
               "--out", SWEEP_OUT, "--device", "cuda"]
    if EVAL_SUBJECT == "0007":
        command.append("--allow-training-subjects")  # desarrollo, no evidencia
    subprocess.run(command, check=True)

    sweep = json.loads(Path(SWEEP_OUT).read_text())
    s = sweep["summary"]
    print(f"\n=== SWEEP ({EVAL_SUBJECT}) ===")
    print(f"casos                        {s['num_cases']}")
    print(f"responde (cambia)            {s['responds_fraction']:.1%}   R medio {s['mean_responsiveness']:.4f}")
    print(f"dominio destino correcto     {s['correct_target_fraction']:.1%}   (chance {s['chance_correct_target_fraction']:.1%})")
    print(f"margen medio de clasificación {s['mean_classification_margin']:+.4f}")
    print(f"rank medio del pedido        {s['mean_rank_of_requested']:.2f} de 5")
    print("\npor contraste:")
    for contrast, block in sweep["by_contrast"].items():
        print(f"  {contrast:10s} responde {block['responds_fraction']:5.1%}  "
              f"correcto {block['correct_target_fraction']:5.1%}  "
              f"margen {block['mean_classification_margin']:+.4f}")
else:
    sweep = None
    print("SKIP sweep: falta SB_CKPT.")

## 6. Calidad del coupling por contraste

¿El coupling `nn` le entrega peores pseudo-pares a T2/T2-FLAIR que a T1w? Se mide en el **mismo
espacio de descriptores** que usa el coupling, así que es lo que el coupling realmente vio, no un
proxy.

`retrieval_advantage` cerca de 1.0 = el vecino recuperado no está más cerca que un miembro
cualquiera del pool, o sea el coupling es efectivamente aleatorio para ese contraste.

In [ ]:
COUPLING_OUT = f"{WORK}/gate0_coupling_{EVAL_SUBJECT}.json"

command = ["python", "-u", "-m", "fieldbridge.cli", "gate0-coupling-quality",
           "--bank-dir", LATENTS, "--split-json", SPLIT_RESPLIT,
           "--subjects", EVAL_SUBJECT, "--pool-size", "4", "--nn-candidates", "5",
           "--out", COUPLING_OUT, "--device", "cuda"]
subprocess.run(command, check=True)

coupling = json.loads(Path(COUPLING_OUT).read_text())
print(f"\n=== COUPLING ({EVAL_SUBJECT}) ===")
print(f"{'contraste':12s} {'nearest':>9s} {'pool median':>12s} {'advantage':>10s}")
for contrast, block in coupling["by_contrast"].items():
    print(f"{contrast:12s} {block['mean_nearest_distance']:9.4f} "
          f"{block['mean_pool_median_distance']:12.4f} {block['retrieval_advantage']:10.3f}")

## 7. Residual — descriptivo, no veredicto autónomo

Residual = `z_target_real − afin(z_source)` sobre los 120 pares con supervisión real (0007 y
0006; **0009 excluido por config**, el CLI aborta si aparece).

**Lo que este número NO es.** La fracción predecible es un coseno vóxel a vóxel entre dos
cerebros distintos. Error de registro y variación anatómica la atenúan aunque exista un efecto
transferible. El gate mide esa atenuación (coseno cross-sujeto mismo dominio, **0.345** en este
banco) y reporta el residual contra ella como **escala de referencia descriptiva** — no como
techo de aprendibilidad, y fuera de la regla de decisión. Un valor bajo es compatible tanto con
"no hay efecto transferible" como con "hay uno que un producto interno fijo no ve".

n = 2. Es un indicador, no un estadístico. Es **una entrada** del Gate 0, no la decisión.

In [ ]:
RESIDUAL_OUT = f"{WORK}/gate0_residual.json"

subprocess.run(["python", "-u", "-m", "fieldbridge.cli", "gate0-residual-gate",
                "--bank-dir", LATENTS, "--split-json", SPLIT_RESPLIT,
                "--subjects", "0006", "0007",
                "--affine-baseline", AFFINE_ALL, AFFINE_FG,
                "--out", RESIDUAL_OUT, "--device", "cuda"], check=True)

residual = json.loads(Path(RESIDUAL_OUT).read_text())
alignment = next(iter(residual["baselines"].values()))["predictability"]["anatomical_alignment"]
print(f"\n=== RESIDUAL ===")
print(f"pares {residual['num_pairs']}   piso f16 {residual['quantization_floor_energy']:.3e}")
print(f"escala de referencia por alineación: cos {alignment['mean_cosine']:.4f} "
      f"-> {alignment['predictable_fraction_reference_scale']:.4f}\n")
header = (f"{'baseline':12s} {'E_identity':>11s} {'E_residual':>11s} {'explicado':>10s} "
          f"{'E_anatomía':>11s} {'predecible':>11s} {'vs escala':>10s}")
print(header); print("-" * len(header))
for name, block in residual["baselines"].items():
    o, p = block["overall"], block["predictability"]
    print(f"{name:12s} {o['identity_energy']:11.5f} {o['residual_energy']:11.5f} "
          f"{o['explained_fraction']:9.1%} {block['anatomy_floor']['mean_energy']:11.5f} "
          f"{p['median_predictable_fraction']:11.4f} "
          f"{p['median_predictable_fraction_vs_reference_scale']:9.1%}")
print("\nveredicto (una entrada, no la decisión):", residual["verdict"]["decision"])
print(residual["verdict"]["caveat"])

## 8. PARA — inspecciona antes de gastar A100

Todo lo anterior fue barato. Antes de encender el paso caro, contesta:

- ¿El sweep muestra clasificación de dominio destino por encima del azar (0.20)?
- ¿El coupling es igual de bueno en los tres contrastes, o `retrieval_advantage` ≈ 1 en T2/FLAIR?
- ¿El afín latente es ≈ identidad (esperado), y por tanto el baseline que importa es el de
  intensidad en imagen?

Si esas respuestas ya cierran la pregunta, **no corras el paso 9**. Si dejan algo sin resolver
—típicamente: los baselines fotométricos igualan o no a SB— entonces sí, y ahí se justifica.

## 9. Gate de referencias (CARO, ~1-1.5 h de A100)

Mismo protocolo que v2 (60 pares, decode full-volume, Heun 20 pasos), con LPIPS y la columna
diagnóstica de SSIM post-normalización robusta.

| fila | qué es |
|---|---|
| `identity` | decode del latente source |
| `affine` | afín cerrado en latente |
| `identity_robust_affine` | identity remapeado a la ventana de percentiles del dominio destino |
| `identity_histogram_matched` | identity con la distribución de intensidad completa del destino |
| `sb_v2` | el checkpoint SB v2 |
| `sb_v2_minus_affine_diagnostic` | **descomposición diagnóstica, no un modelo** |
| `ceiling` | decode del latente target real |

Los dos de intensidad no cuestan decode extra: son remapeos fotométricos de un volumen ya
decodificado.

`sb_v2_minus_affine_diagnostic` puede salirse del manifold latente; sus métricas no son un
reclamo sobre ningún sistema desplegable y no se promociona por su score.

`ssim_robust` es **diagnóstico**, separado del SSIM oficial — reescala ambos volúmenes por sus
propios percentiles, así que no es comparable con ningún número del leaderboard.

**Resumible por par:** cada par se escribe a `<out>.pairs.jsonl` al terminar. Si Colab se cae,
re-ejecuta esta celda y sigue donde quedó.

In [ ]:
REFERENCE_OUT = f"{WORK}/gate0_reference_{EVAL_SUBJECT}.json"

if not RUN_EXPENSIVE_REFERENCE:
    reference = None
    print("SALTADO: RUN_EXPENSIVE_REFERENCE = False.")
    print("Enciéndelo a propósito, después de inspeccionar los diagnósticos baratos.")
else:
    command = ["python", "-u", "-m", "fieldbridge.cli", "gate0-reference-gate",
               "--bank-dir", LATENTS, "--split-json", SPLIT_RESPLIT,
               "--subjects", EVAL_SUBJECT,
               "--affine-baseline", AFFINE_FG,
               "--intensity-baseline", INTENSITY_AFF, INTENSITY_HIST,
               "--vae-config", VAE_CONFIG, "--vae-checkpoint", VAE_CKPT,
               "--metrics", "ssim", "nrmse", "lpips",
               "--solver", "heun", "--n-steps", "20",
               "--out", REFERENCE_OUT, "--resume", "--device", "cuda"]
    if HAS_SB:
        command += ["--transport-config", SB_CONFIG, "--transport-checkpoint", SB_CKPT]
    if EVAL_SUBJECT == "0007":
        command.append("--allow-training-subjects")
    subprocess.run(command, check=True)

    reference = json.loads(Path(REFERENCE_OUT).read_text())
    assert reference["decode"]["path_used"] == ["full"], \
        f"STOP: decode cayó a {reference['decode']['path_used']} (aproximación tiled)."
    print("\ndecode:", reference["decode"])
    print("roles de métrica:", reference["metric_roles"])

## 10. Reporte: tablas estratificadas + tabla por contraste

In [ ]:
def table(block, method_names, title):
    lines = [f"**{title}** ({block['num_pairs']} pares)", "",
             "| referencia | nRMSE | SSIM | SSIM robusto (diag.) | LPIPS |", "|---|---|---|---|---|"]
    for name in method_names:
        m = block["methods"][name]
        cell = lambda key: f"{m[key]:.4f}" if key in m else "—"
        lines.append(f"| {name} | {cell('nrmse')} | {cell('ssim')} | {cell('ssim_robust')} | {cell('lpips')} |")
    return "\n".join(lines) + "\n"

report = [f"# Gate 0 — resultados ({EVAL_SUBJECT})\n",
          f"Commit `{HEAD}`. Contrato: `{WORK}/gate0_contract.json`.\n"]

if reference is not None:
    names, strata = reference["method_names"], reference["strata"]
    report += [
        table(reference["overall"], names, "Agregado"),
        table(strata["catastrophic_identity"], names,
              f"Estrato catastrófico ({strata['catastrophic_identity']['definition']}, "
              f"{strata['catastrophic_identity']['fraction_of_pairs']:.1%} de los pares)"),
        table(strata["ordinary"], names, "Estrato ordinario"),
    ]
    for contrast, block in reference["by_contrast"].items():
        report.append(table(block, names, f"Contraste {contrast}"))
else:
    report.append("_Gate de referencias no ejecutado (RUN_EXPENSIVE_REFERENCE = False)._\n")

# Tabla por contraste desde todos los diagnósticos disponibles.
DIAGNOSTICS_OUT = f"{WORK}/gate0_per_contrast_{EVAL_SUBJECT}.json"
command = ["python", "-m", "fieldbridge.cli", "gate0-report",
           "--residual", RESIDUAL_OUT, "--coupling", COUPLING_OUT, "--out", DIAGNOSTICS_OUT]
if sweep is not None:
    command += ["--sweep", SWEEP_OUT]
if reference is not None:
    command += ["--reference", REFERENCE_OUT]
subprocess.run(command, check=True, stdout=subprocess.DEVNULL)

per_contrast = json.loads(Path(DIAGNOSTICS_OUT).read_text())

def cell(payload, key, fmt="{:.4f}"):
    if isinstance(payload, dict) and isinstance(payload.get(key), (int, float)):
        return fmt.format(payload[key])
    return "—"

report.append("## Por contraste\n")
report.append("| contraste | ceiling SSIM | identity SSIM | coupling advantage "
              "| dominio correcto | chance | residual explicado |")
report.append("|---|---|---|---|---|---|---|")
for contrast, entry in per_contrast["by_contrast"].items():
    conditioning = entry.get("target_conditioning", {})
    report.append(
        f"| {contrast} "
        f"| {cell(entry.get('stage1_ceiling'), 'ssim')} "
        f"| {cell(entry.get('identity'), 'ssim')} "
        f"| {cell(entry.get('coupling'), 'retrieval_advantage', '{:.3f}')} "
        f"| {cell(conditioning, 'correct_target_fraction', '{:.1%}')} "
        f"| {cell(conditioning, 'chance_correct_target_fraction', '{:.1%}')} "
        f"| {cell(entry.get('residual'), 'explained_fraction', '{:.1%}')} |"
    )
report.append(f"\n_No medido:_ {per_contrast['not_measured']['loss_contribution']}\n")

report.append("## Residual (descriptivo)\n")
report.append(f"- lectura mecánica del umbral: **{residual['verdict']['decision']}**")
report.append(f"- {residual['verdict']['caveat']}\n")

text = "\n".join(report)
Path(f"{WORK}/GATE0_REPORT.md").write_text(text, encoding="utf-8")
print(text)